In [10]:
# Import and set up

In [20]:
import boto3
import sagemaker
import pandas as pd
import os
from sagemaker.inputs import TrainingInput
from sagemaker import image_uris, get_execution_role
from sklearn.metrics import (
    accuracy_score, classification_report,
    roc_auc_score, confusion_matrix
)
 
session     = sagemaker.Session()
role        = get_execution_role()
bucket      = "prognostica-cancer-project"
prefix      = "lung-cancer"
S3_FEATURES = f"s3://{bucket}/features/"
s3_client   = boto3.client("s3")
 
print(f"Role   : {role}")
print(f"Bucket : {bucket}")
print(f"Source : {S3_FEATURES}")
 

Role   : arn:aws:iam::381491860224:role/LabRole
Bucket : prognostica-cancer-project
Source : s3://prognostica-cancer-project/features/


In [21]:
# Verify S3 files exist before doing anything

In [22]:
TRAIN_URI = S3_FEATURES + "lung_train.csv"
VAL_URI   = S3_FEATURES + "lung_val.csv"
TEST_URI  = S3_FEATURES + "lung_test.csv"
 
def verify_s3_uri(uri):
    bkt, key = uri.replace("s3://", "").split("/", 1)
    try:
        s3_client.head_object(Bucket=bkt, Key=key)
        print(f"✅ Found: {uri}")
    except Exception:
        raise FileNotFoundError(f"\n❌ NOT FOUND: {uri}\n   → Re-run data_preparation_handoff.ipynb first.")
 
print("Verifying S3 paths...")
verify_s3_uri(TRAIN_URI)
verify_s3_uri(VAL_URI)
verify_s3_uri(TEST_URI)
 

Verifying S3 paths...
✅ Found: s3://prognostica-cancer-project/features/lung_train.csv
✅ Found: s3://prognostica-cancer-project/features/lung_val.csv
✅ Found: s3://prognostica-cancer-project/features/lung_test.csv


In [23]:
# CELL 3 — Download, reformat & re-upload for SageMaker
#   SageMaker built-in XGBoost requires:
#     • No header row
#     • Target column FIRST

In [33]:
os.makedirs("data", exist_ok=True)
 
def download_s3_uri(uri, local_path):
    bkt, key = uri.replace("s3://", "").split("/", 1)
    s3_client.download_file(bkt, key, local_path)
    print(f"Downloaded → {local_path}")
 
def prepare_and_upload(local_path, s3_key, target_col="lung_cancer"):
    df = pd.read_csv(local_path)
 
    # Encode YES/NO → 1/0 if not already numeric
    if df[target_col].dtype == object:
        df[target_col] = df[target_col].map({"YES": 1, "NO": 0})
 
    # Move target to first column
    cols = [target_col] + [c for c in df.columns if c != target_col]
    df   = df[cols]
 
    out = local_path.replace(".csv", "_ready.csv")
    df.to_csv(out, index=False, header=False)       # No header row
    s3_client.upload_file(out, bucket, s3_key)
    print(f"Uploaded  → s3://{bucket}/{s3_key}  ({len(df)} rows)")
    return df
 
print("\nDownloading from S3...")
download_s3_uri(TRAIN_URI, "data/lung_train.csv")
download_s3_uri(VAL_URI,   "data/lung_val.csv")
download_s3_uri(TEST_URI,  "data/lung_test.csv")
 
print("\nReformatting & re-uploading for SageMaker...")
train_df = prepare_and_upload("data/lung_train.csv", f"{prefix}/train/lung_train.csv")
val_df   = prepare_and_upload("data/lung_val.csv",   f"{prefix}/val/lung_val.csv")
test_df  = prepare_and_upload("data/lung_test.csv",  f"{prefix}/test/lung_test.csv")
 
train_input = TrainingInput(f"s3://{bucket}/{prefix}/train/", content_type="text/csv")
val_input   = TrainingInput(f"s3://{bucket}/{prefix}/val/",   content_type="text/csv")
 
print("\n✅ Data ready for training.")


Downloaded → data/lung_train.csv
Downloaded → data/lung_val.csv
Downloaded → data/lung_test.csv

Reformatting & re-uploading for SageMaker...
Uploaded  → s3://prognostica-cancer-project/lung-cancer/train/lung_train.csv  (13699 rows)
Uploaded  → s3://prognostica-cancer-project/lung-cancer/val/lung_val.csv  (2936 rows)
Uploaded  → s3://prognostica-cancer-project/lung-cancer/test/lung_test.csv  (2936 rows)

✅ Data ready for training.


In [42]:
# Convert all boolean columns to integers (0/1)
for df in [train_df, val_df, test_df]:
    bool_cols = df.select_dtypes(include='bool').columns.tolist()
    df[bool_cols] = df[bool_cols].astype(int)
    if bool_cols:
        print(f"Converted boolean cols: {bool_cols}")

# Re-upload test features
test_features = test_df.iloc[:, 1:]
test_features.to_csv("data/lung_test_features.csv", index=False, header=False)
s3_client.upload_file(
    "data/lung_test_features.csv", bucket,
    f"{prefix}/test_features/lung_test_features.csv"
)

# Verify
sample = pd.read_csv("data/lung_test_features.csv", header=None, nrows=2)
print("\nSample (must be all numbers):")
print(sample)


Sample (must be all numbers):
   0   1   2   3   4   5   6   7   8   9   10  11  12  13  14  15  16  17  18
0   1  59   1   1   2   1   1   2   2   1   2   1   2   2   2   0   1   0   0
1   1  76   2   1   1   1   1   2   1   2   2   1   1   2   2   0   0   0   1


In [43]:
# CELL 4 — Handle class imbalance & define XGBoost estimator


In [26]:
# Your data is ~87% YES / 13% NO - scale_pos_weight corrects for this
yes_count        = (train_df.iloc[:, 0] == 1).sum()
no_count         = (train_df.iloc[:, 0] == 0).sum()
scale_pos_weight = round(no_count / yes_count, 4)
print(f"scale_pos_weight = {scale_pos_weight}  (NO: {no_count} / YES: {yes_count})")
 
# Use generic Estimator with built-in XGBoost container image
# This avoids the entry_point requirement entirely
container = image_uris.retrieve(
    framework="xgboost",
    region=session.boto_region_name,
    version="1.7-1",        # Fall back to "1.5-1" if this raises an error
)
print(f"Container image: {container}")
 
xgb = sagemaker.estimator.Estimator(
    image_uri=container,
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    output_path=f"s3://{bucket}/{prefix}/output",
    sagemaker_session=session,
    hyperparameters={
        "objective":             "binary:logistic",
        "eval_metric":           "auc",
        "scale_pos_weight":      scale_pos_weight,
        "num_round":             200,
        "max_depth":             5,
        "min_child_weight":      5,
        "eta":                   0.1,
        "gamma":                 1,
        "subsample":             0.8,
        "colsample_bytree":      0.8,
        "early_stopping_rounds": 20,
    },
)
print("Estimator defined successfully.")

scale_pos_weight = 0.1504  (NO: 1791 / YES: 11908)
Container image: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1
Estimator defined successfully.


In [27]:
# CELL 5 — Train


In [28]:
xgb.fit(
    inputs={"train": train_input, "validation": val_input},
    logs=True,   # Stream CloudWatch logs to notebook
    wait=True,   # Block until job completes
)
 
print("\n✅ Training complete!")
print(f"Model artifact → {xgb.model_data}")

INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-03-22-18-38-05-624


2026-03-22 18:38:06 Starting - Starting the training job...
2026-03-22 18:38:41 Downloading - Downloading input data...
2026-03-22 18:39:06 Downloading - Downloading the training image......
2026-03-22 18:39:57 Training - Training image download completed. Training in progress../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-03-22 18:40:04.364 ip-10-2-248-172.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-03-22 18:40:04.433 ip-10-2-248-172.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-03-22:18:40:04:INFO] Imported framework sagemaker_xgboost_container.training
[2026-03-22:18:40:04:INFO] Failed to parse hyperparameter eval_metr

In [36]:
import boto3

logs = boto3.client("logs", region_name=session.boto_region_name)

# Get the latest transform job log
log_group = "/aws/sagemaker/TransformJobs"
streams = logs.describe_log_streams(
    logGroupName=log_group,
    orderBy="LastEventTime",
    descending=True,
    limit=3
)

for stream in streams["logStreams"]:
    print(f"\n--- {stream['logStreamName']} ---")
    events = logs.get_log_events(
        logGroupName=log_group,
        logStreamName=stream["logStreamName"],
        limit=30
    )
    for e in events["events"]:
        print(e["message"])


--- sagemaker-xgboost-2026-03-22-19-00-29-311/i-0faf6975bdeb6e9d6-1774206223/data-log ---
2026-03-22T19:06:08.159:[sagemaker logs]: MaxConcurrentTransforms=4, MaxPayloadInMB=6, BatchStrategy=MULTI_RECORD
2026-03-22T19:06:08.407:[sagemaker logs]: prognostica-cancer-project/lung-cancer/test_features/lung_test_features.csv: ClientError: 415
2026-03-22T19:06:08.407:[sagemaker logs]: prognostica-cancer-project/lung-cancer/test_features/lung_test_features.csv: 
2026-03-22T19:06:08.407:[sagemaker logs]: prognostica-cancer-project/lung-cancer/test_features/lung_test_features.csv: Message:
2026-03-22T19:06:08.407:[sagemaker logs]: prognostica-cancer-project/lung-cancer/test_features/lung_test_features.csv: Loading csv data failed with Exception, please ensure data is in csv format:
2026-03-22T19:06:08.407:[sagemaker logs]: prognostica-cancer-project/lung-cancer/test_features/lung_test_features.csv:  <class 'ValueError'>
2026-03-22T19:06:08.407:[sagemaker logs]: prognostica-cancer-project/lung-

In [45]:
# Re-upload train and val with boolean fix
for df, name, key in [
    (train_df, "train", f"{prefix}/train/lung_train.csv"),
    (val_df,   "val",   f"{prefix}/val/lung_val.csv"),
]:
    out = f"data/lung_{name}_ready.csv"
    df.to_csv(out, index=False, header=False)
    s3_client.upload_file(out, bucket, key)
    print(f"Re-uploaded {name}: {df.shape[0]} rows x {df.shape[1]} cols")

print("\nAll clean - run Cell 6 now!")

Re-uploaded train: 13699 rows x 20 cols
Re-uploaded val: 2936 rows x 20 cols

All clean - run Cell 6 now!


In [30]:
#cell 6 Batch Transform (score the test set)

In [44]:
test_features = test_df.iloc[:, 1:]
 
# Save with no header and no index - overwrite any cached version
test_features.to_csv("data/lung_test_features.csv", index=False, header=False)
 
# Delete old S3 object first to avoid stale cached file
try:
    s3_client.delete_object(Bucket=bucket, Key=f"{prefix}/test_features/lung_test_features.csv")
    print("Deleted old test features from S3.")
except Exception:
    pass
 
s3_client.upload_file(
    "data/lung_test_features.csv", bucket,
    f"{prefix}/test_features/lung_test_features.csv"
)
print(f"Uploaded fresh test features: {test_features.shape[0]} rows x {test_features.shape[1]} cols, no header.")
 
# Verify first row looks numeric (no column names)
sample = pd.read_csv("data/lung_test_features.csv", header=None, nrows=2)
print("Sample (should be all numbers):")
print(sample)
 
transformer = xgb.transformer(
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/predictions",
    assemble_with="Line",
    accept="text/csv",
)
 
transformer.transform(
    data=f"s3://{bucket}/{prefix}/test_features/",
    content_type="text/csv",
    split_type="Line",
    wait=True,
    logs=True,
)
 
print("\nBatch transform complete!")

INFO:sagemaker:Creating model with name: sagemaker-xgboost-2026-03-22-19-21-19-160


Uploaded fresh test features: 2936 rows x 19 cols, no header.
Sample (should be all numbers):
   0   1   2   3   4   5   6   7   8   9   10  11  12  13  14  15  16  17  18
0   1  59   1   1   2   1   1   2   2   1   2   1   2   2   2   0   1   0   0
1   1  76   2   1   1   1   1   2   1   2   2   1   1   2   2   0   0   0   1


INFO:sagemaker:Creating transform job with name: sagemaker-xgboost-2026-03-22-19-21-19-915


............................./miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-03-22:19:26:07:INFO] No GPUs detected (normal if no gpus installed)
[2026-03-22:19:26:07:INFO] No GPUs detected (normal if no gpus installed)
[2026-03-22:19:26:07:INFO] nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or 

In [ ]:
# CELL 7 - Evaluate

In [46]:
s3_client.download_file(
    bucket,
    f"{prefix}/predictions/lung_test_features.csv.out",
    "data/lung_predictions.csv"
)
 
probs  = pd.read_csv("data/lung_predictions.csv", header=None)[0]
preds  = (probs >= 0.5).astype(int)
labels = test_df.iloc[:, 0].reset_index(drop=True)
 
print("\nEvaluation Results")
print(f"Accuracy : {accuracy_score(labels, preds):.4f}")
print(f"ROC-AUC  : {roc_auc_score(labels, probs):.4f}")
 
print("\nConfusion Matrix:")
print(pd.DataFrame(
    confusion_matrix(labels, preds),
    index=["Actual NO", "Actual YES"],
    columns=["Pred NO", "Pred YES"]
))
 
print("\nClassification Report:")
print(classification_report(labels, preds, target_names=["NO (0)", "YES (1)"]))


Evaluation Results
Accuracy : 0.4404
ROC-AUC  : 0.4957

Confusion Matrix:
            Pred NO  Pred YES
Actual NO       222       162
Actual YES     1481      1071

Classification Report:
              precision    recall  f1-score   support

      NO (0)       0.13      0.58      0.21       384
     YES (1)       0.87      0.42      0.57      2552

    accuracy                           0.44      2936
   macro avg       0.50      0.50      0.39      2936
weighted avg       0.77      0.44      0.52      2936

